# Exploracion y limpieza: suppliers

Perfilado de Bronze y comparacion de proveedores antes/despues.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from transformacion.pipeline_transformacion import CLEANERS
from pyspark.sql import functions as F
ENTITY = 'suppliers'
storage_account = os.getenv('STORAGE_ACCOUNT_NAME', 'stecommercedirty01')
df_raw = spark.read.option('multiLine', True).json(f'abfss://bronze@{storage_account}.dfs.core.windows.net/{ENTITY}/')
print(f'Entidad: {ENTITY} | filas: {df_raw.count():,}')
df_raw.printSchema()

In [ ]:
df_raw.limit(20).display()
df_raw.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df_raw.columns]).display()

In [ ]:
df_clean = CLEANERS[ENTITY](df_raw, spark)
pairs = [(c, f'{c}_clean') for c in df_raw.columns if f'{c}_clean' in df_clean.columns]
for raw_name, clean_name in pairs:
    print(f'ANTES vs DESPUES: {raw_name}')
    df_clean.select(F.col(raw_name).alias('antes'), F.col(clean_name).alias('despues')).display()
df_clean.printSchema()

In [ ]:
df_clean.select([F.sum((F.col(raw) != F.col(clean)).cast('int')).alias(raw) for raw, clean in pairs]).display() if pairs else None